# Clasificación Multiclase del Nivel de Obesidad con Regresión Softmax

**Proyecto Integrador — Unidad 1: Fundamentos y Modelos Lineales en Machine Learning**

Este notebook implementa el experimento completo de clasificación multiclase (7 clases) usando **Regresión Logística Softmax (multinomial)**, siguiendo el enfoque Google de formulación (ver `FormulacionProblema_ClasificaciónMúltiple.pdf`).

**Caso de negocio:** sistema de alerta temprana y prevención de riesgo nutricional laboral, que clasifica a una persona en una de 7 categorías de estado nutricional a partir de sus hábitos alimenticios, actividad física y datos demográficos.

**Dataset:** [Estimation of Obesity Levels Based On Eating Habits and Physical Condition](https://archive.ics.uci.edu/dataset/544) (UCI Machine Learning Repository).


In [ ]:
# 1. Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report)

RANDOM_STATE = 42
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)


## 2. Carga de datos

In [ ]:
DATA_PATH = "dataset_obesidad.csv"

df = pd.read_csv(DATA_PATH)
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
df.head()


## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Estructura general del dataset

In [ ]:
df.info()


### 3.2 Descripción de los features

Diccionario de variables: nombre, tipo, significado de negocio.

In [ ]:
dictionary = pd.DataFrame([
    ("Age", "Numérica continua", "Edad de la persona (años)"),
    ("Gender", "Binaria", "Sexo (Male/Female)"),
    ("Height", "Numérica continua", "Estatura (metros)"),
    ("Weight", "Numérica continua", "Peso (kg)"),
    ("family_history_with_overweight", "Binaria", "Antecedente familiar de sobrepeso"),
    ("FAVC", "Binaria", "Consumo frecuente de alimentos hipercalóricos"),
    ("FCVC", "Numérica (1-3)", "Frecuencia de consumo de vegetales"),
    ("NCP", "Numérica continua", "Número de comidas principales al día"),
    ("CAEC", "Ordinal", "Consumo de alimentos entre comidas"),
    ("SMOKE", "Binaria", "Fuma"),
    ("CH2O", "Numérica continua", "Consumo diario de agua (litros)"),
    ("SCC", "Binaria", "Monitorea las calorías que consume"),
    ("FAF", "Numérica continua", "Frecuencia de actividad física semanal"),
    ("TUE", "Numérica continua", "Tiempo de uso de dispositivos tecnológicos"),
    ("CALC", "Ordinal", "Frecuencia de consumo de alcohol"),
    ("MTRANS", "Categórica nominal", "Medio de transporte habitual"),
    ("NObeyesdad", "Categórica (target, 7 clases)", "Nivel de obesidad — variable objetivo"),
], columns=["Variable", "Tipo", "Descripción"])
dictionary


La variable objetivo (`NObeyesdad`) tiene 7 clases ordenadas por severidad clínica:
`Insufficient_Weight → Normal_Weight → Overweight_Level_I → Overweight_Level_II → Obesity_Type_I → Obesity_Type_II → Obesity_Type_III`.

### 3.3 Datos faltantes, duplicados e inconsistencias (calidad de datos)

In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())
print()
n_dupes = df.duplicated().sum()
print(f"Registros duplicados exactos: {n_dupes}")


In [ ]:
# Se eliminan los duplicados exactos: no aportan información nueva y, de mantenerse,
# podrían repartirse entre train y test generando fuga de datos (data leakage).
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_after = len(df)
print(f"Registros antes: {n_before} | después de eliminar duplicados: {n_after}")


### 3.4 Análisis de desbalance de clases

In [ ]:
class_order = ["Insufficient_Weight","Normal_Weight","Overweight_Level_I","Overweight_Level_II",
               "Obesity_Type_I","Obesity_Type_II","Obesity_Type_III"]

class_counts = df["NObeyesdad"].value_counts()
print(class_counts)

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nRazón de desbalance (mayoría/minoría): {imbalance_ratio:.2f}")

plt.figure(figsize=(7,4))
class_counts.reindex(class_order).plot(kind="bar", color="steelblue")
plt.ylabel("N° de registros"); plt.title("Distribución de clases (NObeyesdad)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


El desbalance es **leve** (razón ≈1.3, no severo). Aun así, se mitiga con dos estrategias:
1. **Partición estratificada** train/test para preservar la proporción de clases.
2. **`class_weight="balanced"`** en la Regresión Logística, que pondera el costo de cada clase de forma inversamente proporcional a su frecuencia.

### 3.5 Estadística descriptiva (general y por clase)

In [ ]:
numeric_cols = ["Age","Height","Weight","FCVC","NCP","CH2O","FAF","TUE"]
df[numeric_cols].describe().T.round(2)


In [ ]:
# Estadística descriptiva de Peso por clase — ilustra la separabilidad de clases
df.groupby("NObeyesdad")["Weight"].describe().reindex(class_order).round(1)


### 3.6 Análisis multivariado — correlación

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matriz de correlación (variables numéricas)")
plt.tight_layout()
plt.show()


## 4. Preprocesamiento: codificación de variables categóricas

- Variables **binarias** (Gender, family_history_with_overweight, FAVC, SMOKE, SCC) → 0/1.
- Variables **ordinales** (CAEC, CALC: no/A veces/Frecuentemente/Siempre) → escala 0-3, respetando su orden natural.
- Variable **nominal** (MTRANS) → one-hot encoding, al no tener un orden natural.

In [ ]:
target_col = "NObeyesdad"
class_to_idx = {c: i for i, c in enumerate(class_order)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
y = df[target_col].map(class_to_idx).to_numpy()

X = df.drop(columns=[target_col]).copy()

binary_cols = ["Gender", "family_history_with_overweight", "FAVC", "SMOKE", "SCC"]
for c in binary_cols:
    X[c] = X[c].map({"no": 0, "yes": 1, "Female": 0, "Male": 1})

ordinal_map = {"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3}
X["CAEC"] = X["CAEC"].map(ordinal_map)
X["CALC"] = X["CALC"].map(ordinal_map)

X = pd.get_dummies(X, columns=["MTRANS"], drop_first=False)
feature_names = X.columns.tolist()
X.head()


## 5. División del dataset: entrenamiento y prueba (80/20, estratificada)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

print(f"Entrenamiento: {X_train.shape[0]} muestras")
print(f"Prueba: {X_test.shape[0]} muestras")


## 6. Escalado de variables

Al ser la Regresión Logística Softmax un modelo **lineal** (`z = θᵀx`), las variables numéricas continuas deben escalarse para que ninguna domine el entrenamiento por su magnitud. El escalador se ajusta **solo con train** y se aplica a test, evitando *data leakage*.

In [ ]:
scale_cols = numeric_cols
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])


## 7. Entrenamiento: Regresión Logística Softmax + exploración de hiperparámetros

`LogisticRegression` con el solver `lbfgs` ajusta automáticamente una **Softmax multinomial** cuando hay más de dos clases (en vez de descomponerlo en varios problemas binarios One-vs-Rest). Se explora el hiperparámetro de regularización `C` (inverso de la fuerza de regularización L2) con validación cruzada estratificada de 5 folds, optimizando F1-macro.

In [ ]:
param_grid = {"C": [0.01, 0.1, 1, 10, 100]}

base_model = LogisticRegression(
    solver="lbfgs", max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(base_model, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_
print("Mejor C:", grid.best_params_["C"])
print("F1-macro (CV):", round(grid.best_score_, 4))

cv_table = pd.DataFrame({
    "C": param_grid["C"],
    "F1_macro_CV": grid.cv_results_["mean_test_score"],
})
cv_table


## 8. Resultados del Train

El modelo se evalúa aquí **sobre los mismos datos con los que fue entrenado**, solo como diagnóstico de ajuste (se compara con test en la sección 10).

In [ ]:
def eval_split(model, Xs, ys):
    pred = model.predict(Xs)
    acc = accuracy_score(ys, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(ys, pred, average="macro", zero_division=0)
    cm = confusion_matrix(ys, pred)
    report = classification_report(ys, pred, target_names=class_order, zero_division=0, output_dict=True)
    return {"accuracy": acc, "precision_macro": prec, "recall_macro": rec, "f1_macro": f1,
            "confusion_matrix": cm, "report": report}

def plot_confusion(cm, title):
    plt.figure(figsize=(7,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_order, yticklabels=class_order)
    plt.xlabel("Predicho"); plt.ylabel("Real"); plt.title(title)
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

train_metrics = eval_split(best_model, X_train_scaled, y_train)
print(f"Accuracy (train): {train_metrics['accuracy']:.4f}")
print(f"Precisión macro (train): {train_metrics['precision_macro']:.4f}")
print(f"Recall macro (train): {train_metrics['recall_macro']:.4f}")
print(f"F1-macro (train): {train_metrics['f1_macro']:.4f}")
plot_confusion(train_metrics["confusion_matrix"], "Matriz de confusión — Train")


## 9. Resultados del Test

In [ ]:
test_metrics = eval_split(best_model, X_test_scaled, y_test)
print(f"Accuracy (test): {test_metrics['accuracy']:.4f}")
print(f"Precisión macro (test): {test_metrics['precision_macro']:.4f}")
print(f"Recall macro (test): {test_metrics['recall_macro']:.4f}")
print(f"F1-macro (test): {test_metrics['f1_macro']:.4f}")
plot_confusion(test_metrics["confusion_matrix"], "Matriz de confusión — Test")


In [ ]:
# Tabla comparativa train vs. test
comparison = pd.DataFrame({
    "Métrica": ["Accuracy", "Precisión macro", "Recall macro", "F1-macro"],
    "Train": [train_metrics["accuracy"], train_metrics["precision_macro"],
              train_metrics["recall_macro"], train_metrics["f1_macro"]],
    "Test": [test_metrics["accuracy"], test_metrics["precision_macro"],
             test_metrics["recall_macro"], test_metrics["f1_macro"]],
}).round(4)
comparison


In [ ]:
# Métricas por clase (test)
report_df = pd.DataFrame(test_metrics["report"]).T.loc[class_order, ["precision","recall","f1-score","support"]].round(4)
report_df


## 10. Importancia relativa de las variables

In [ ]:
coefs = np.abs(best_model.coef_).mean(axis=0)
coef_series = pd.Series(coefs, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(8,6))
coef_series.head(12).sort_values().plot(kind="barh", color="teal")
plt.title("Importancia relativa de features (|coeficiente| promedio entre clases)")
plt.tight_layout()
plt.show()

coef_series.head(10)


**Observación importante:** `Weight` y `Height` dominan ampliamente la importancia del modelo. Esto es esperable: las categorías de `NObeyesdad` están definidas, en la práctica, por rangos de Índice de Masa Corporal (IMC = Peso / Talla²). El modelo completo aprende en gran parte a reproducir esa fórmula. Esto se explora más a fondo en la sección 12 (Ablación).

## 11. Prueba con features propios

Se construyen tres observaciones sintéticas representativas de los perfiles descritos en el problema de negocio, y se interpreta la coherencia de la predicción con el dominio.

In [ ]:
custom_samples = pd.DataFrame([
    {  # Perfil A: oficinista sedentario, alto consumo calórico, poca actividad
        "Age": 34, "Gender": "Male", "Height": 1.75, "Weight": 98,
        "family_history_with_overweight": "yes", "FAVC": "yes", "FCVC": 2, "NCP": 3,
        "CAEC": "Sometimes", "SMOKE": "no", "CH2O": 1.5, "SCC": "no", "FAF": 0, "TUE": 2,
        "CALC": "Sometimes", "MTRANS": "Automobile",
    },
    {  # Perfil B: alta actividad física, restricción alimentaria, bajo peso
        "Age": 24, "Gender": "Female", "Height": 1.70, "Weight": 47,
        "family_history_with_overweight": "no", "FAVC": "no", "FCVC": 3, "NCP": 2,
        "CAEC": "no", "SMOKE": "no", "CH2O": 2.5, "SCC": "yes", "FAF": 3, "TUE": 0,
        "CALC": "no", "MTRANS": "Walking",
    },
    {  # Perfil C: hábitos equilibrados
        "Age": 29, "Gender": "Female", "Height": 1.65, "Weight": 62,
        "family_history_with_overweight": "no", "FAVC": "no", "FCVC": 3, "NCP": 3,
        "CAEC": "Sometimes", "SMOKE": "no", "CH2O": 2, "SCC": "no", "FAF": 2, "TUE": 1,
        "CALC": "no", "MTRANS": "Public_Transportation",
    },
])

Xc = custom_samples.copy()
for c in binary_cols:
    Xc[c] = Xc[c].map({"no": 0, "yes": 1, "Female": 0, "Male": 1})
Xc["CAEC"] = Xc["CAEC"].map(ordinal_map)
Xc["CALC"] = Xc["CALC"].map(ordinal_map)
Xc = pd.get_dummies(Xc, columns=["MTRANS"])
Xc = Xc.reindex(columns=feature_names, fill_value=0)
Xc[scale_cols] = scaler.transform(Xc[scale_cols])

proba = best_model.predict_proba(Xc)
pred_idx = best_model.predict(Xc)

for i, perfil in enumerate(["A", "B", "C"]):
    top3 = np.argsort(proba[i])[::-1][:3]
    print(f"Perfil {perfil} -> Predicción: {idx_to_class[pred_idx[i]]}")
    for j in top3:
        print(f"   {class_order[j]:22s} {proba[i][j]:.4f}")
    print()


Las tres predicciones son coherentes con el dominio: el perfil sedentario de alto consumo calórico (A) se clasifica en una categoría de obesidad, el perfil de alta actividad y restricción alimentaria (B) en peso insuficiente, y el perfil de hábitos equilibrados (C) en peso normal.

## 12. Discusión de los resultados

### 12.1 Comparación train vs. test — over/underfitting

La brecha entre accuracy y F1-macro de train y test es mínima (ver tabla de la sección 9), lo que indica que el modelo **generaliza correctamente**, sin overfitting ni underfitting relevantes: la regularización L2 elegida por validación cruzada logra un balance adecuado entre sesgo y varianza.

### 12.2 Análisis de la matriz de confusión

Los errores se concentran casi exclusivamente entre clases **adyacentes** en la escala de severidad (p. ej. Normal↔Overweight_I, Overweight_II↔Obesity_I), nunca entre extremos opuestos. Esto es coherente con que las fronteras de IMC entre categorías son continuas: los casos cercanos a un umbral son los más difíciles de separar para un modelo lineal.

## 13. Experimento: ¿qué pasa sin Height/Weight?

Para separar el efecto puramente antropométrico (IMC) del efecto de los hábitos de vida, se entrena un segundo modelo Softmax **sin** `Height` y `Weight`.

In [ ]:
drop_cols = ["Height", "Weight"]
X_train_abl = X_train_scaled.drop(columns=drop_cols)
X_test_abl = X_test_scaled.drop(columns=drop_cols)

model_abl = LogisticRegression(
    solver="lbfgs", max_iter=2000, class_weight="balanced",
    C=grid.best_params_["C"], random_state=RANDOM_STATE
)
model_abl.fit(X_train_abl, y_train)
abl_metrics = eval_split(model_abl, X_test_abl, y_test)

print("Modelo completo   -> Accuracy test:", round(test_metrics["accuracy"], 4),
      "| F1-macro test:", round(test_metrics["f1_macro"], 4))
print("Modelo sin H/W    -> Accuracy test:", round(abl_metrics["accuracy"], 4),
      "| F1-macro test:", round(abl_metrics["f1_macro"], 4))


Aun sin `Height`/`Weight`, el modelo supera ampliamente el azar (1/7 ≈ 14.3%), lo que confirma que los **hábitos sí contienen señal predictiva real**, aunque considerablemente menor que cuando se dispone del IMC directamente. Esto tiene una implicación de negocio importante: si la empresa ya mide peso y talla, no necesita Machine Learning para calcular la categoría (es una fórmula cerrada); el verdadero valor de un sistema de **alerta temprana** está en un modelo que use solo hábitos, para anticipar el riesgo *antes* de que el cambio de peso ya haya ocurrido.

## 14. Conclusiones

1. El modelo de Regresión Logística Softmax, entrenado únicamente con train y con regularización L2 calibrada por validación cruzada, generaliza correctamente al test (brecha train-test mínima en accuracy y F1-macro), sin señales de sobreajuste.
2. Los errores de clasificación se concentran entre categorías de severidad adyacente, nunca entre extremos opuestos, lo que confirma que el modelo captura el orden subyacente del problema.
3. `Weight` y `Height` dominan la importancia del modelo porque las clases están definidas a partir del IMC; el experimento de ablación muestra que, sin estas variables, el desempeño cae de forma sustancial pero se mantiene muy por encima del azar.
4. Para el caso de negocio (alerta temprana), el modelo completo es útil como **clasificador del estado actual**, mientras que el modelo reducido (solo hábitos) es la base más honesta para una **alerta predictiva real**, ya que no depende de una medición que solo confirma un estado que ya ocurrió.
5. **Limitación del enfoque lineal:** al ser un modelo lineal, no captura interacciones complejas entre hábitos (p. ej. el efecto combinado de baja actividad física y alto consumo calórico podría no ser simplemente aditivo). Como recomendación de mejora, se sugiere comparar este resultado con modelos no lineales (árboles de decisión, ensambles) sobre el subconjunto de solo-hábitos, y complementar siempre las predicciones con revisión de personal de salud ocupacional antes de cualquier acción sobre un empleado.
